# Errors and Debugging

The most common type of errors at runtime stems from worker tasks.
A first measure to preventing this is using the correct types, but still something might go wrong.
In this example we will produce an error and investigate it with the debugging tools in tierkreis.

## Worker Errors

Worker errors can occur in multiple ways.
For python workers an error occurs when an uncaught exception raises.
For other workers (including python) a non-zero exit code will also produce an error.

Defining a graph that will always run an error:

In [ ]:
from error_worker import fail
from tierkreis.builder import Graph, Workflow
from tierkreis.controller.data.core import EmptyModel
from tierkreis.controller.data.models import TKR


def error_graph() -> Workflow:
    g = Graph(EmptyModel, TKR[str])
    output = g.task(fail())
    return g.finish_with_outputs(output)

The task `fail` raises an error while running. `wait_for()` reports a failed workflow as `ValueError`; the chained exception and notes contain the worker details.


In [ ]:
from tierkreis import new_default

runtime = await new_default()
try:
    with runtime:
        workflow_id = await runtime.save_workflow("error_handling", error_graph())
        run_id = await runtime.start_new_run(workflow_id, {"value": "world!"})
        await runtime.wait_for(run_id, 0)
except ValueError as error:
    print(error)

states = await runtime.debug_read_node_states(run_id, 0, ["N0"])
assert states["N0"].status == "Error"


## Debugging

Runtime logging is controlled through its logging configuration. `debug_read_node_states()` can inspect selected node locations after a run completes or fails.


In [ ]:
states = await runtime.debug_read_node_states(run_id, 0, ["N0"])
print(states["N0"].status)


```{warning}
Interactive Python-worker breakpoints and reading detailed errors directly from the legacy controller storage are not exposed by the Rust runtime binding yet.
```


In [ ]:
# See the compatibility note above for the current debugging limitations.
